In [0]:
# ======================================
# Dataset: olist_sellers
# Layer: Silver
# Source: Bronze (Delta)
# Grain: 1 row per seller_id
# ======================================

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
%run ../commons/silver_utils

In [0]:
%run ../../00_config/paths

In [0]:
df_bronze = spark.read.format("delta").load(BRONZE_SELLERS_PATH)
df_bronze.printSchema()

In [0]:
df = normalize_columns(df_bronze)

In [0]:
df = df.select(
    F.col("seller_id"),
    F.col("seller_zip_code_prefix"),
    F.trim(F.lower(F.col("seller_city"))).alias("seller_city"),
    F.upper(F.col("seller_state")).alias("seller_state")
)

In [0]:
df = df.filter(F.col("seller_id").isNotNull())

In [0]:
df = df.filter(F.length(F.col("seller_zip_code_prefix")) >= 4)

In [0]:
df = df.filter(F.length(F.col("seller_state")) == 2)

In [0]:
print("Bronze count:", df_bronze.count())
print("Silver count:", df.count())

In [0]:
df.printSchema()

In [0]:
df.limit(10).display()

In [0]:
write_silver(df, SILVER_SELLERS_PATH)